In [1]:
from pyspark.sql import SparkSession
import pyspark
import os
from dotenv import load_dotenv
import traceback
from pathlib import Path

# If we are in the 'notebooks' folder, go up one level
if Path.cwd().name == 'scripts':
    os.chdir('../..')

print(f"Current working directory: {os.getcwd()}")

load_dotenv()

Current working directory: /home/mounir_salam/python_projects/1000Genome_Analysis


True

In [ ]:
!wget https://ftp-trace.ncbi.nih.gov/1000genomes/ftp/sequence.index

In [ ]:
!wget -O data/input/populations.tsv https://ftp-trace.ncbi.nih.gov/1000genomes/ftp/20131219.populations.tsv

In [2]:
print(os.getenv('GCS_CONNECTOR_VERSION'))
print(os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))

hadoop3-2.2.22
keys/de-zoomcamp-key.json


In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("GCP-Connect") \
    .config("spark.jars.packages", f"com.google.cloud.bigdataoss:gcs-connector:{os.getenv('GCS_CONNECTOR_VERSION')}") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/18 22:59:44 WARN Utils: Your hostname, MounirLenovo2, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/18 22:59:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/mounir_salam/python_projects/1000Genome_Analysis/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/mounir_salam/.ivy2.5.2/cache
The jars for the packages stored in: /home/mounir_salam/.ivy2.5.2/jars
com.google.cloud.bigdataoss#gcs-connector added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a77dfeaf-0014-4a89-a0d8-841376b3ab93;1.0
	confs: [default]
	found com.google.cloud.bigdataoss#gcs-connector;hadoop3-2.2.22 in central
	found com.google.api-client#google-api-client-jackson2;2.0.1 in central
	found com.

In [ ]:
spark.stop()

In [ ]:
# Download file using requests stream

import requests
from src.db_connection.builder import IOBuilder

connector = IOBuilder.get_resource("gcs_raw_data")
iterator = connector.client.list_buckets()

for bucket in iterator:
    print(bucket)

print("Current Bucket", connector.get_bucket())


In [ ]:
from tqdm import tqdm

url = "https://ftp-trace.ncbi.nih.gov/1000genomes/ftp/sequence.index"
blob = connector.get_blob("raw/sequence.index")

# Check if file already exists
if not blob.exists():
    print("FIle does not exist, downloading...")

    with requests.get(url, stream=True) as r:
        r.raise_for_status()

        total_size = int(r.headers.get('Content-Length', 0))
        blob.chunk_size = 5 * 1024 * 1024  # Use 5MB chunks

        with tqdm.wrapattr(
            r.raw, "read",
            total=total_size,
            desc="Uploading raw/sequence.index"
        ) as buffered_stream:
            blob.upload_from_file(buffered_stream, content_type=r.headers.get('content-type'), rewind=False)
else:
    print("File already exists")

In [4]:
url = "https://ftp-trace.ncbi.nih.gov/1000genomes/ftp/20131219.populations.tsv"
blob = connector.get_blob("raw/populations.tsv")

# Check if file already exists
if not blob.exists():
    print("FIle does not exist, downloading...")

    with requests.get(url, stream=True) as r:
        r.raise_for_status()

        total_size = int(r.headers.get('Content-Length', 0))
        blob.chunk_size = 5 * 1024 * 1024  # Use 5MB chunks

        with tqdm.wrapattr(
            r.raw, "read",
            total=total_size,
            desc="Uploading raw/populations.tsv"
        ) as buffered_stream:
            blob.upload_from_file(buffered_stream, content_type=r.headers.get('content-type'), rewind=False)
else:
    print("File already exists")

NameError: name 'connector' is not defined

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, VarcharType, IntegerType, TimestampType, BooleanType

schema = StructType([
    StructField('FASTQ_FILE', StringType(), True), 
    StructField('MD5', StringType(), True), 
    StructField('RUN_ID', StringType(), True), 
    StructField('STUDY_ID', StringType(), True), 
    StructField('STUDY_NAME', StringType(), True), 
    StructField('CENTER_NAME', StringType(), True), 
    StructField('SUBMISSION_ID', StringType(), True), 
    StructField('SUBMISSION_DATE', TimestampType(), True), 
    StructField('SAMPLE_ID', StringType(), True), 
    StructField('SAMPLE_NAME', StringType(), True), 
    StructField('POPULATION', StringType(), True), 
    StructField('EXPERIMENT_ID', StringType(), True), 
    StructField('INSTRUMENT_PLATFORM', StringType(), True), 
    StructField('INSTRUMENT_MODEL', StringType(), True), 
    StructField('LIBRARY_NAME', StringType(), True), 
    StructField('RUN_NAME', StringType(), True), 
    StructField('RUN_BLOCK_NAME', StringType(), True), 
    StructField('INSERT_SIZE', IntegerType(), True), 
    StructField('LIBRARY_LAYOUT', StringType(), True), 
    StructField('PAIRED_FASTQ', StringType(), True), 
    StructField('WITHDRAWN', IntegerType(), True), 
    StructField('WITHDRAWN_DATE', TimestampType(), True), 
    StructField('COMMENT', StringType(), True), 
    StructField('READ_COUNT', IntegerType(), True), 
    StructField('BASE_COUNT', IntegerType(), True), 
    StructField('ANALYSIS_GROUP', StringType(), True)
])

In [6]:
# tab separated
sequence = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(
    "data/input/sequence.index",
    sep="\t"
)

In [8]:
sequence.rdd.getNumPartitions()

from pyspark.sql.functions import spark_partition_id

sequence.withColumn("partition_id", spark_partition_id()) \
  .groupBy("partition_id") \
  .count() \
  .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|16019|
|           1|16712|
|           2|16223|
|           3|16310|
|           4|16129|
|           5|15799|
|           6|16612|
|           7|17249|
|           8|17457|
|           9|17232|
|          10|16934|
|          11| 5044|
+------------+-----+



In [12]:
from pyspark.sql import functions as F

# set columns to lower case
sequence = sequence.select([F.col(c).alias(c.lower()) for c in sequence.columns])

# Set missing md5 to null (currently looking like this .........................)
sequence.withColumn("md5", F.when(F.col("md5").rlike(r"\.+"), None) .otherwise(F.col("md5"))).filter(F.col("md5").isNull()).count()


9834

In [ ]:
sequence_df = sequence.toPandas()

In [46]:
# Upload to postgres local
import pandas as pd
from src.db_connection.builder import IOBuilder

connector = IOBuilder.get_resource("local_pg")
sequence_df.columns = [c.lower() for c in sequence_df.columns]
sequence_df.head()
sequence_df.to_sql("sequence", connector.engine, index=False)

720

In [9]:
from pyspark.sql.functions import col, length
sequence.cache

<bound method DataFrame.cache of DataFrame[FASTQ_FILE: string, MD5: string, RUN_ID: string, STUDY_ID: string, STUDY_NAME: string, CENTER_NAME: string, SUBMISSION_ID: string, SUBMISSION_DATE: timestamp, SAMPLE_ID: string, SAMPLE_NAME: string, POPULATION: string, EXPERIMENT_ID: string, INSTRUMENT_PLATFORM: string, INSTRUMENT_MODEL: string, LIBRARY_NAME: string, RUN_NAME: string, RUN_BLOCK_NAME: string, INSERT_SIZE: int, LIBRARY_LAYOUT: string, PAIRED_FASTQ: string, WITHDRAWN: int, WITHDRAWN_DATE: timestamp, COMMENT: string, READ_COUNT: int, BASE_COUNT: int, ANALYSIS_GROUP: string]>

In [42]:
sequence.show()

+--------------------+--------------------+---------+---------+--------------------+-----------+-------------+-------------------+---------+-----------+----------+-------------+-------------------+--------------------+--------------+-----------------+--------------+-----------+--------------+--------------------+---------+--------------+-------+----------+----------+--------------+
|          FASTQ_FILE|                 MD5|   RUN_ID| STUDY_ID|          STUDY_NAME|CENTER_NAME|SUBMISSION_ID|    SUBMISSION_DATE|SAMPLE_ID|SAMPLE_NAME|POPULATION|EXPERIMENT_ID|INSTRUMENT_PLATFORM|    INSTRUMENT_MODEL|  LIBRARY_NAME|         RUN_NAME|RUN_BLOCK_NAME|INSERT_SIZE|LIBRARY_LAYOUT|        PAIRED_FASTQ|WITHDRAWN|WITHDRAWN_DATE|COMMENT|READ_COUNT|BASE_COUNT|ANALYSIS_GROUP|
+--------------------+--------------------+---------+---------+--------------------+-----------+-------------+-------------------+---------+-----------+----------+-------------+-------------------+--------------------+------------

In [14]:
sequence.schema

StructType([StructField('FASTQ_FILE', StringType(), True), StructField('MD5', StringType(), True), StructField('RUN_ID', StringType(), True), StructField('STUDY_ID', StringType(), True), StructField('STUDY_NAME', StringType(), True), StructField('CENTER_NAME', StringType(), True), StructField('SUBMISSION_ID', StringType(), True), StructField('SUBMISSION_DATE', TimestampType(), True), StructField('SAMPLE_ID', StringType(), True), StructField('SAMPLE_NAME', StringType(), True), StructField('POPULATION', StringType(), True), StructField('EXPERIMENT_ID', StringType(), True), StructField('INSTRUMENT_PLATFORM', StringType(), True), StructField('INSTRUMENT_MODEL', StringType(), True), StructField('LIBRARY_NAME', StringType(), True), StructField('RUN_NAME', StringType(), True), StructField('RUN_BLOCK_NAME', StringType(), True), StructField('INSERT_SIZE', IntegerType(), True), StructField('LIBRARY_LAYOUT', StringType(), True), StructField('PAIRED_FASTQ', StringType(), True), StructField('WITHDR

In [15]:
# Create table view
sequence.createOrReplaceTempView("sequence")

In [36]:
spark.sql("""
    SELECT
        s1.sample_id_1,
        s1.sample_name,
        s2.sample_id_2,
        s2.sample_name
    FROM (
        SELECT
            upper(s1.sample_id) as sample_id_1,
            s1.sample_name
        FROM sequence as s1
        GROUP BY 1,2
    ) as s1
    JOIN (
        SELECT
            upper(s2.sample_id) as sample_id_2,
            s2.sample_name
        FROM sequence as s2
        GROUP BY 1,2
    ) as s2
    ON s1.sample_name = s2.sample_name
    WHERE s1.sample_id_1 <> s2.sample_id_2 AND s1.sample_name = s2.sample_name
""").show()

+-----------+-----------+-----------+-----------+
|sample_id_1|sample_name|sample_id_2|sample_name|
+-----------+-----------+-----------+-----------+
|  SRS008545|    HG00308|  SRS024887|    HG00308|
|  SRS001262|    NA11933|  SRS173375|    NA11933|
|  SRS001261|    NA11932|  SRS173374|    NA11932|
|  SRS173375|    NA11933|  SRS001262|    NA11933|
|  SRS173374|    NA11932|  SRS001261|    NA11932|
|  SRS000082|    NA12812|  SRS139092|    NA12812|
|  SRS000079|    NA12762|  SRS114396|    NA12762|
|  SRS024887|    HG00308|  SRS008545|    HG00308|
|  SRS114396|    NA12762|  SRS000079|    NA12762|
|  SRS139092|    NA12812|  SRS000082|    NA12812|
+-----------+-----------+-----------+-----------+



In [41]:
spark.sql("""
    SELECT 
        sample_name, 
        COUNT(DISTINCT sample_id) as id_count,
        LISTAGG(sample_id, ',') as associated_ids
    FROM sequence
    GROUP BY sample_name
    HAVING COUNT(DISTINCT sample_id) > 1;
""").show()

+-----------+--------+--------------------+
|sample_name|id_count|      associated_ids|
+-----------+--------+--------------------+
|    HG00308|       2|SRS008545,SRS0085...|
|    NA11932|       2|SRS001261,SRS0012...|
|    NA11933|       2|SRS001262,SRS0012...|
|    NA12762|       2|SRS000079,SRS0000...|
|    NA12812|       2|SRS000082,SRS0000...|
+-----------+--------+--------------------+

